In [ ]:
from __future__ import annotations

import arrow
import polars as pl
import torch
import yfinance

from diffsmile.config import dataset_config

# Download SPX and VIX data
# take some dates from before for niceness I guess.
start_date = arrow.get(dataset_config.train_start).shift(days=-7).format("YYYY-MM-DD")
end_date = arrow.get(dataset_config.val_end).shift(days=+7).format("YYYY-MM-DD")


raw_data = yfinance.download(["^GSPC", "^VIX"], start=start_date, end=end_date)["Close"]
df = pl.from_pandas(raw_data.reset_index()).rename({"^GSPC": "spx", "^VIX": "vix"})

surfaces_path = dataset_config.merged_surfaces_path
all_surfaces_dates = torch.load(surfaces_path, weights_only=False)["dates"]

# Calculate returns and squared returns
df = df.with_columns(
    [
        ((pl.col("spx").shift(-1) / pl.col("spx")) - 1).alias("r_next"),
        (pl.col("spx") / pl.col("spx").shift(1) - 1).alias("r"),
        (pl.col("vix") / pl.col("vix").shift(1) - 1).alias("vix_ret"),
    ]
).drop_nulls()

df = df.with_columns((pl.col("r") ** 2).alias("r2"))

# Compute EWMA components and VIX return
conditioning = df.select(
    [
        "Date",
        pl.col("r_next").alias("ret"),
        pl.col("r").ewm_mean(alpha=0.156, adjust=False).alias("ewm_short"),
        pl.col("r").ewm_mean(alpha=0.118, adjust=False).alias("ewm_long"),
        pl.col("r2").ewm_mean(alpha=0.3, adjust=False).alias("ewm2_short"),
        pl.col("r2").ewm_mean(alpha=0.15, adjust=False).alias("ewm2_long"),
        pl.col("vix_ret"),
    ]
).with_columns(pl.col("Date").cast(pl.Date))

# drop following dates as we are missing those for some reason...
conditioning_filtered = conditioning.filter(pl.col("Date").is_in(all_surfaces_dates))

print(conditioning_filtered)

conditioning_tensor = torch.tensor(conditioning_filtered.to_numpy(), dtype=torch.float32)
torch.save(conditioning_tensor, dataset_config.output_conditioning_scalars)
print(conditioning_tensor.shape)

[*********************100%***********************]  2 of 2 completed

shape: (3_499, 7)
┌────────────┬───────────┬───────────┬───────────┬────────────┬───────────┬───────────┐
│ Date       ┆ ret       ┆ ewm_short ┆ ewm_long  ┆ ewm2_short ┆ ewm2_long ┆ vix_ret   │
│ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---        ┆ ---       ┆ ---       │
│ date       ┆ f64       ┆ f64       ┆ f64       ┆ f64        ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╪═══════════╪════════════╪═══════════╪═══════════╡
│ 2010-01-04 ┆ 0.003116  ┆ 0.000359  ┆ -0.000096 ┆ 0.000099   ┆ 0.000053  ┆ -0.075646 │
│ 2010-01-05 ┆ 0.000546  ┆ 0.000789  ┆ 0.000283  ┆ 0.000072   ┆ 0.000046  ┆ -0.034431 │
│ 2010-01-06 ┆ 0.004001  ┆ 0.000751  ┆ 0.000314  ┆ 0.000051   ┆ 0.000039  ┆ -0.009819 │
│ 2010-01-07 ┆ 0.002882  ┆ 0.001258  ┆ 0.000749  ┆ 0.00004    ┆ 0.000036  ┆ -0.005219 │
│ 2010-01-08 ┆ 0.001747  ┆ 0.001511  ┆ 0.001001  ┆ 0.000031   ┆ 0.000032  ┆ -0.048793 │
│ …          ┆ …         ┆ …         ┆ …         ┆ …          ┆ …         ┆ …         │
│ 2023-12-22 ┆